# Day-14 Activity Features — Does *Which* Activity Happened Help?

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

## 1. Goal

Notebook 07 used seven **count** features from the first 14 days (number of events, unique activities, and so on).
Logistic Regression did not beat the majority baseline.

This notebook asks: does knowing **which activities** were recorded by day 14 add useful information to those counts?

To keep the comparison fair, I use **exactly the same** cases, target, day-14 cutoff, seven features and train/test split
as notebook 07, and only add new activity features.

## 2. Load the Data and Rebuild Notebook 07's Cohort

Same rules as notebook 07:

- **Completed case:** `case:endDate` is present.
- **Duration:** last event minus first event, in days.
- **Day-14 cutoff:** `prediction_time` = first event + 14 days.
- **Eligible:** the case's last event is **after** `prediction_time` (still active at day 14).
- **Target:** `long_case = 1` if `duration_days > 63`, otherwise 0.

In [1]:
import os
import warnings

os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "false"  # no progress bar while loading
warnings.filterwarnings("ignore")                 # hide an optional speed-up tip from PM4Py

import pandas as pd
import pm4py
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




*When PM4Py is imported it prints a short welcome and licence message. This is normal and not an error.*

In [2]:
log = pm4py.read_xes("../data/raw/BPIC15_1.xes")
df = pm4py.convert_to_dataframe(log)

case_data = df.groupby("case:concept:name").agg(
    first_event=("time:timestamp", "min"),
    last_event=("time:timestamp", "max"),
    end_date=("case:endDate", "first"),
)
case_data["duration_days"] = (case_data["last_event"] - case_data["first_event"]).dt.total_seconds() / (24 * 60 * 60)
case_data = case_data[case_data["end_date"].notna()]   # completed cases only

case_data["prediction_time"] = case_data["first_event"] + pd.Timedelta(days=14)
active = case_data[case_data["last_event"] > case_data["prediction_time"]]

print("Completed cases:", len(case_data))
print("Cases still active at day 14:", len(active))

Completed cases: 956
Cases still active at day 14: 917


## 3. Events Recorded by Day 14

Every event in this log has `lifecycle:transition = "complete"`, so each row is one **recorded** activity occurrence.
The features below therefore describe which activity codes were **recorded** by day 14.

The helper below keeps only the events of the eligible cases that are timestamped **on or before** each case's
`prediction_time`. The cutoffs come from the `active` table, so they stay fixed.

In [3]:
print(df["lifecycle:transition"].value_counts())

lifecycle:transition
complete    52217
Name: count, dtype: int64


In [4]:
def events_by_day14(event_table, cases):
    # keep only events of eligible cases that happened on or before that case's day-14 cutoff
    events = event_table[event_table["case:concept:name"].isin(cases.index)].copy()
    events["prediction_time"] = events["case:concept:name"].map(cases["prediction_time"])
    events["first_event"] = events["case:concept:name"].map(cases["first_event"])
    first14 = events[events["time:timestamp"] <= events["prediction_time"]].copy()
    first14["day"] = (first14["time:timestamp"] - first14["first_event"]).dt.total_seconds() / (24 * 60 * 60)
    return first14


first14 = events_by_day14(df, active)
print("Events recorded by day 14:", len(first14))
print("Latest event used, in days after the case start:", first14["day"].max())

Events recorded by day 14: 10930
Latest event used, in days after the case start: 14.0


## 4. The Original Seven Features (Same Code as Notebook 07)

In [5]:
count_columns = [
    "events_first14",
    "unique_activities_first14",
    "unique_resources_first14",
    "repeated_activities_first14",
    "events_first7",
    "events_days8_14",
    "days_since_last_event",
]


def count_features(first14, cases):
    # the seven count features from notebook 07
    first14 = first14.copy()
    first14["in_first7"] = first14["day"] <= 7
    features = first14.groupby("case:concept:name").agg(
        events_first14=("concept:name", "size"),
        unique_activities_first14=("concept:name", "nunique"),
        unique_resources_first14=("org:resource", "nunique"),
        events_first7=("in_first7", "sum"),
        last_event_by_day14=("time:timestamp", "max"),
    )
    features["repeated_activities_first14"] = features["events_first14"] - features["unique_activities_first14"]
    features["events_days8_14"] = features["events_first14"] - features["events_first7"]
    features["days_since_last_event"] = (
        cases.loc[features.index, "prediction_time"] - features["last_event_by_day14"]
    ).dt.total_seconds() / (24 * 60 * 60)
    return features[count_columns]


features = count_features(first14, active)
features["long_case"] = (active.loc[features.index, "duration_days"] > 63).astype(int)

print("Cases with features:", len(features))
print(features["long_case"].value_counts())

Cases with features: 917
long_case
1    472
0    445
Name: count, dtype: int64


## 5. Rebuild Notebook 07's Train/Test Split

I use the same code as notebook 07, on a table with the same rows in the same order (one row per case, sorted by case ID).

To check that the split really is the same, I re-run notebook 07's model (Logistic Regression without scaling) on this
split. If the split is identical, the metrics and the confusion matrix must match notebook 07 exactly
(accuracy 0.49, precision 0.50, recall 0.63, F1 0.56; confusion matrix 30 / 59 / 35 / 60).

In [6]:
X7 = features[count_columns]
y = features["long_case"]

X7_train, X7_test, y_train, y_test = train_test_split(
    X7,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
train_ids = X7_train.index
test_ids = X7_test.index

print("Training cases:", len(train_ids), "| long:", y_train.sum(), "| not long:", (y_train == 0).sum())
print("Test cases:", len(test_ids), "| long:", y_test.sum(), "| not long:", (y_test == 0).sum())
print("Overlap between train and test:", len(set(train_ids) & set(test_ids)))

Training cases: 733 | long: 377 | not long: 356
Test cases: 184 | long: 95 | not long: 89
Overlap between train and test: 0


In [7]:
check_model = LogisticRegression(max_iter=1000).fit(X7_train, y_train)   # notebook 07's model, unscaled
check_pred = check_model.predict(X7_test)

print(f"Accuracy {accuracy_score(y_test, check_pred):.2f} | Precision {precision_score(y_test, check_pred):.2f} | "
      f"Recall {recall_score(y_test, check_pred):.2f} | F1 {f1_score(y_test, check_pred):.2f}")
print("Confusion matrix:", confusion_matrix(y_test, check_pred).ravel().tolist())
assert confusion_matrix(y_test, check_pred).ravel().tolist() == [30, 59, 35, 60], "split does not match notebook 07"
print("Notebook 07's result is reproduced exactly on this split.")

Accuracy 0.49 | Precision 0.50 | Recall 0.63 | F1 0.56
Confusion matrix: [30, 59, 35, 60]
Notebook 07's result is reproduced exactly on this split.


## 6. Select Activities Using Training Cases Only

For each activity code (`concept:name`), I calculate the percentage of **training** cases in which it was recorded by day 14.
I keep activities recorded in **10%–90%** of training cases (very rare and almost universal activities are not useful), and
take at most the **15 most frequent**. Ties are broken alphabetically by activity code, so the selection is always the same.

The target is **not** used here.

In [8]:
train_first14 = first14[first14["case:concept:name"].isin(train_ids)]

activity_share = train_first14.groupby("concept:name")["case:concept:name"].nunique() / len(train_ids) * 100
activity_share = activity_share.rename("train_cases_pct").reset_index()

candidates = activity_share[(activity_share["train_cases_pct"] >= 10) & (activity_share["train_cases_pct"] <= 90)]
candidates = candidates.sort_values(["train_cases_pct", "concept:name"], ascending=[False, True])
selected = candidates.head(15)

print("Activity codes recorded by day 14 in training cases:", len(activity_share))
print("Codes in the 10–90% range:", len(candidates))
print("Codes selected:", len(selected))

Activity codes recorded by day 14 in training cases: 130
Codes in the 10–90% range: 20
Codes selected: 15


The table shows the original activity code and the English activity name as it is recorded in the log (`activityNameEN`).
The same English name can belong to more than one code.

In [9]:
# the English label recorded most often for each selected activity code
english_names = []
for activity in selected["concept:name"]:
    labels = train_first14.loc[train_first14["concept:name"] == activity, "activityNameEN"]
    english_names.append(labels.mode()[0])

selected_table = selected[["concept:name"]].copy()
selected_table["activityNameEN"] = english_names
selected_table["train_cases_pct"] = selected["train_cases_pct"]
selected_table.round(1).reset_index(drop=True)

,concept:name,activityNameEN,train_cases_pct
0,01_HOOFD_015,phase application received,88.8
1,01_HOOFD_020,send confirmation receipt,88.8
2,01_HOOFD_030_1,send confirmation receipt,81.9
3,01_HOOFD_030_2,enter senddate acknowledgement,80.9
4,01_HOOFD_011,OLO messaging active,62.3
5,02_DRZ_010,forward to the competent authority,61.0
6,04_BPT_005,regular procedure without MER,60.0
7,01_HOOFD_101,registration date publication,59.8
8,01_HOOFD_061,start WABOprocedure,46.4
9,01_HOOFD_065_2,enter senddate procedure confirmation,42.6


## 7. Create Binary Activity Features

One feature per selected activity: **1** if the activity was recorded by day 14, otherwise **0**.
A 0 means "not recorded by day 14", not "will never happen".

In [10]:
selected_activities = selected["concept:name"].tolist()


def activity_flags(first14, activities, case_ids):
    # 1 if the activity was recorded by day 14 in that case, otherwise 0
    flags = pd.DataFrame(index=case_ids)
    for activity in activities:
        cases_with_activity = first14.loc[first14["concept:name"] == activity, "case:concept:name"].unique()
        flags["act_" + activity] = flags.index.isin(cases_with_activity).astype(int)
    return flags


flags = activity_flags(first14, selected_activities, features.index)
features = features.join(flags)
activity_columns = flags.columns.tolist()

print("Activity features:", len(activity_columns))
features[activity_columns].head()

Activity features: 15


,act_01_HOOFD_015,act_01_HOOFD_020,act_01_HOOFD_030_1,act_01_HOOFD_030_2,act_01_HOOFD_011,act_02_DRZ_010,act_04_BPT_005,act_01_HOOFD_101,act_01_HOOFD_061,act_01_HOOFD_065_2,act_01_HOOFD_065_1,act_01_HOOFD_065_0,act_01_HOOFD_100_0,act_01_HOOFD_040,act_01_HOOFD_060
case:concept:name,,,,,,,,,,,,,,,
10009138,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0
10051383,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0
10053042,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0
10093171,1,1,1,1,1,1,1,1,1,0,0,1,1,0,0
10186644,1,1,1,0,1,1,1,1,1,0,0,1,1,0,0


## 8. Train and Compare Three Approaches

All three are trained on the same training cases and tested on the same 184 test cases.

- **A. Majority baseline:** always predicts the most common class in the training labels.
- **B. Logistic Regression, 7 count features.**
- **C. Logistic Regression, 7 count features + activity features.**

B and C use the same simple pipeline: `StandardScaler` + `LogisticRegression(max_iter=1000)`, default threshold 0.5.
The scaler is fitted on the training data only, as part of the pipeline.

In [11]:
X_train_7, X_test_7 = features.loc[train_ids, count_columns], features.loc[test_ids, count_columns]
X_train_all = features.loc[train_ids, count_columns + activity_columns]
X_test_all = features.loc[test_ids, count_columns + activity_columns]

model_a = DummyClassifier(strategy="most_frequent").fit(X_train_7, y_train)
model_b = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train_7, y_train)
model_c = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train_all, y_train)

print("Majority class in training labels:", model_a.predict(X_test_7)[0])

Majority class in training labels: 1


In [12]:
def evaluate(model, X_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]   # probability of class 1 (long case)
    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(y_test, predictions),
        "Recall": recall_score(y_test, predictions),
        "F1": f1_score(y_test, predictions),
        "ROC-AUC": roc_auc_score(y_test, probabilities),
    }


results = pd.DataFrame({
    "A. Majority baseline": evaluate(model_a, X_test_7),
    "B. LogReg, 7 count features": evaluate(model_b, X_test_7),
    "C. LogReg, 7 counts + activities": evaluate(model_c, X_test_all),
}).T
results.round(2)

,Accuracy,Precision,Recall,F1,ROC-AUC
A. Majority baseline,0.52,0.52,1.00,0.68,0.50
"B. LogReg, 7 count features",0.49,0.50,0.63,0.56,0.53
"C. LogReg, 7 counts + activities",0.58,0.60,0.54,0.57,0.59


**ROC-AUC** uses the predicted probabilities and measures how well a model ranks long cases above not-long cases: 0.50 is no better than chance and 1.00 is perfect.

The baseline gives every case the same probability, so its ROC-AUC is 0.50 (no ranking at all).

In [13]:
for name, model, X_test in [("B. 7 count features", model_b, X_test_7), ("C. 7 counts + activities", model_c, X_test_all)]:
    print(name)
    print(pd.DataFrame(
        confusion_matrix(y_test, model.predict(X_test)),
        index=["actual 0 (not long)", "actual 1 (long)"],
        columns=["predicted 0", "predicted 1"],
    ))
    print()

B. 7 count features
                     predicted 0  predicted 1
actual 0 (not long)           30           59
actual 1 (long)               35           60

C. 7 counts + activities
                     predicted 0  predicted 1
actual 0 (not long)           55           34
actual 1 (long)               44           51



## 9. Leakage Check

I change the activity and resource of **every event after each case's day-14 cutoff** to fake values, keeping the cohort
and the cutoffs fixed, and rebuild all features with the same functions. If the features only use information available
by day 14, nothing may change.

In [14]:
changed = df.copy()
cutoff = changed["case:concept:name"].map(active["prediction_time"])
after_day14 = changed["time:timestamp"] > cutoff
changed.loc[after_day14, "concept:name"] = "FAKE_ACTIVITY"
changed.loc[after_day14, "org:resource"] = "FAKE_RESOURCE"

changed_first14 = events_by_day14(changed, active)
changed_features = count_features(changed_first14, active).join(
    activity_flags(changed_first14, selected_activities, features.index)
)

print("Events changed:", after_day14.sum())
print("All features unchanged:", changed_features.equals(features[count_columns + activity_columns]))

Events changed: 32715


All features unchanged: True


## 10. Interpretation

**Same split as notebook 07.** The rebuilt split reproduces notebook 07's result exactly. The scaled model B gives the
same predictions and confusion matrix as notebook 07's unscaled model, so here scaling made no difference to the
predictions.

**Do the activity features improve on the seven count features (C vs B)?** Partly.

- Accuracy rises from 0.49 to 0.58 and precision from 0.50 to 0.60.
- ROC-AUC rises from 0.53 to 0.59, so model C ranks long cases above not-long cases somewhat better.
- F1 stays about the same (0.56 → 0.57) and recall falls (0.63 → 0.54). Model C flags fewer cases as long (85 instead of
  119 of 184): it misses more long cases but raises fewer false alarms (34 instead of 59).

**Do they beat the majority baseline (C vs A)?** On accuracy (0.58 vs 0.52), precision (0.60 vs 0.52) and ROC-AUC (0.59 vs
0.50), yes. On recall and F1, no: the baseline predicts every case as long, so it finds all long cases (recall 1.00,
F1 0.68), but it cannot separate long from not-long cases at all.

**Overall:** knowing *which* activities were recorded by day 14 seems to add a **small amount** of signal that the event
counts alone did not have. The signal is still **weak** (ROC-AUC 0.59, where 0.50 is no better than chance). The test set
has only 184 cases and was already used in notebook 07, so these gains are **exploratory**, not reliable proof that the
model would work on new cases.

## 11. Limitations

- This is an exploratory study of **one Dutch municipality**, not SPAR or Montenegro data.
- The experiment only includes **historically completed cases that were still active at day 14**.
- **Fourteen days** is an experimental choice.
- The **test set is small** (184 cases) and has already been used for exploratory comparisons.
- More complex models are not part of this notebook.